In [1]:
import math
import numpy as np


# ============================================================
# Utilities
# ============================================================

def mittag_leffler_E(alpha: float, beta: float, x: float, tol: float = 1e-13, max_terms: int = 5000) -> float:
    """
    Two-parameter Mittag-Leffler E_{alpha,beta}(x) for real x.
    Power-series evaluation, sufficient here because we only use moderate negative x.
    """
    s = 0.0
    for k in range(max_terms):
        term = (x ** k) / math.gamma(alpha * k + beta)
        s += term
        if abs(term) < tol:
            break
    return s


def phi_kernel(alpha: float, n: int) -> float:
    """
    Base kernel varphi_n from the paper.
    """
    g = math.gamma(1.0 - alpha)
    if n == 1:
        return 1.0 - 1.0 / g
    return (1.0 / g) * ((n - 1) ** (-alpha) - n ** (-alpha))


def build_phi_array(alpha: float, N: int) -> np.ndarray:
    return np.array([0.0] + [phi_kernel(alpha, n) for n in range(1, N + 1)], dtype=float)


# ============================================================
# Continuous side: fractional Riccati
# ============================================================

def solve_fractional_riccati(
    alpha: float,
    gamma: float,
    rho: float,
    nu: float,
    z: complex,
    T: float = 1.0,
    steps: int = 4000,
    blowup_threshold: float = 1e8,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Solve the Volterra form
        h(t) = 1/Gamma(alpha) ∫_0^t (t-s)^(alpha-1) F(h(s)) ds
    with
        F(x) = 0.5(z^2-z) + gamma(rho*nu*z - 1)x + 0.5 (gamma*nu)^2 x^2

    Simple left-point product-integration discretization.
    Good enough for checking whether the solution stays finite.
    """
    dt = T / steps
    times = np.linspace(0.0, T, steps + 1)
    h = np.zeros(steps + 1, dtype=complex)

    c0 = 0.5 * (z * z - z)
    c1 = gamma * (rho * nu * z - 1.0)
    c2 = 0.5 * (gamma * nu) ** 2

    # Precompute kernel increments:
    # ∫_{t_j}^{t_{j+1}} (t_n-s)^(alpha-1) ds / Gamma(alpha)
    # = ((n-j)^alpha - (n-j-1)^alpha) * dt^alpha / Gamma(alpha+1)
    coeff_scale = (dt ** alpha) / math.gamma(alpha + 1.0)

    for n in range(1, steps + 1):
        total = 0.0 + 0.0j
        for j in range(n):
            f_j = c0 + c1 * h[j] + c2 * h[j] * h[j]
            w = ((n - j) ** alpha - (n - j - 1) ** alpha) * coeff_scale
            total += w * f_j
        h[n] = total

        if not np.isfinite(h[n].real) or not np.isfinite(h[n].imag) or abs(h[n]) > blowup_threshold:
            raise RuntimeError(f"Continuous Riccati appears to blow up before T={T}, at step {n}/{steps}.")

    return times, h


# ============================================================
# Discrete side: exact transform recursion
# ============================================================

def beta_from_rho(rho: float) -> float:
    """
    Solve rho = (1-beta)/sqrt(2(1+beta^2)) for beta>1.
    """
    a = 1.0 - 2.0 * rho * rho
    b = -2.0
    c = 1.0 - 2.0 * rho * rho
    # choose root > 1
    disc = b * b - 4.0 * a * c
    roots = [(-b + math.sqrt(disc)) / (2.0 * a), (-b - math.sqrt(disc)) / (2.0 * a)]
    roots = [x for x in roots if x > 1.0]
    if not roots:
        raise ValueError("No admissible beta > 1 found from rho.")
    return roots[0]


def mu_from_nu(theta: float, beta: float, gamma: float, nu: float) -> float:
    """
    From the paper:
        mu = theta(1+beta^2) / (gamma nu^2 (1+beta)^2)
    """
    return theta * (1.0 + beta * beta) / (gamma * nu * nu * (1.0 + beta) ** 2)


def baseline_hat_mu(
    alpha: float,
    gamma: float,
    xi0: float,
    mu: float,
    tau: int,
    N: int,
) -> np.ndarray:
    """
    Build \hat mu_tau(n), n=1,...,N.
    """
    a_tau = 1.0 - gamma * tau ** (-alpha)
    mu_tau = mu * tau ** (alpha - 1.0)

    # base phi
    phi_vals = build_phi_array(alpha, N)
    phi_tau_vals = a_tau * phi_vals

    # cumulative sums of phi_tau
    csum = np.zeros(N + 1, dtype=float)
    for n in range(1, N + 1):
        csum[n] = csum[n - 1] + phi_tau_vals[n]

    hat_mu = np.zeros(N + 1, dtype=float)
    for n in range(1, N + 1):
        s = csum[n - 1]
        hat_mu[n] = mu_tau + xi0 * mu_tau * ((1.0 / (1.0 - a_tau)) * (1.0 - s) - s)

    return hat_mu


def exact_discrete_transform_logphi(
    *,
    alpha: float,
    gamma: float,
    theta: float,
    V0: float,
    rho: float,
    nu: float,
    z: complex,
    T: float,
    tau: int,
) -> tuple[complex, np.ndarray]:
    """
    Compute log Phi^tau(z,T) from the exact recursion in the paper:
        G_n = exp(Theta_+ + sum q_k^+ G_{n-k}) + exp(Theta_- + sum q_k^- G_{n-k}) - 2
        log Phi = sum_m hat mu_tau(m) G_{N-m}

    Returns:
        logPhi, G-array
    """
    if not (0.5 < alpha < 1.0):
        raise ValueError("This routine is for rough case 1/2 < alpha < 1. For alpha=1 use the continuous Heston ODE check.")

    N = int(math.floor(tau * T))

    beta = beta_from_rho(rho)
    xi0 = V0 / theta
    mu = mu_from_nu(theta, beta, gamma, nu)

    a_tau = 1.0 - gamma * tau ** (-alpha)
    eps_tau = 1.0 - a_tau
    c_tau = math.sqrt(theta * eps_tau / (2.0 * mu * tau ** alpha))
    one_minus_exp_neg_c = -math.expm1(-c_tau)
    d_tau = (
        -math.log1p(-(one_minus_exp_neg_c * one_minus_exp_neg_c))
        if c_tau < 0.5
        else c_tau - math.log1p(one_minus_exp_neg_c)
    )

    theta_plus = z * (c_tau - d_tau)
    theta_minus = -z * c_tau

    phi_vals = build_phi_array(alpha, N)
    phi_tau_vals = a_tau * phi_vals
    q_plus = phi_tau_vals / (1.0 + beta)
    q_minus = beta * phi_tau_vals / (1.0 + beta)

    hat_mu = baseline_hat_mu(alpha, gamma, xi0, mu, tau, N)

    G = np.zeros(N + 1, dtype=complex)

    for n in range(0, N + 1):
        s_plus = theta_plus
        s_minus = theta_minus

        if n >= 1:
            # sum_{k=1}^n q_k G_{n-k}
            # note G[0] is needed here
            for k in range(1, n + 1):
                s_plus += q_plus[k] * G[n - k]
                s_minus += q_minus[k] * G[n - k]

        G[n] = np.exp(s_plus) + np.exp(s_minus) - 2.0

        if not np.isfinite(G[n].real) or not np.isfinite(G[n].imag) or abs(G[n]) > 1e12:
            raise RuntimeError(f"Discrete recursion appears unstable at n={n}.")

    logPhi = 0.0 + 0.0j
    for m in range(1, N + 1):
        logPhi += hat_mu[m] * G[N - m]

    return logPhi, G


# ============================================================
# Checks for the paper's parameter sets
# ============================================================

def check_first_rough_case():
    """
    First rough case from the paper:
      alpha=0.62, gamma=0.1, rho=-0.681, nu=0.331, theta=0.3156, V0=0.0392
    """
    alpha = 0.62
    gamma = 0.1
    rho = -0.681
    nu = 0.331
    theta = 0.3156
    V0 = 0.0392
    T = 1.0
    eta = 2.0

    print("=== First rough case ===")
    print(f"eta = {eta}")

    times, h = solve_fractional_riccati(alpha, gamma, rho, nu, eta + 0j, T=T, steps=4000)
    print(f"continuous h(T,eta) = {h[-1]}")

    for tau in [40, 80, 160, 320]:
        logPhi, _ = exact_discrete_transform_logphi(
            alpha=alpha,
            gamma=gamma,
            theta=theta,
            V0=V0,
            rho=rho,
            nu=nu,
            z=eta + 0j,
            T=T,
            tau=tau,
        )
        print(f"tau={tau:3d}, log Phi^tau(eta,T) = {logPhi}")


def check_slice_comparison_case():
    """
    Slice-comparison parameter set from the paper:
      gamma=0.3, rho=-0.7, nu=1, theta=0.02/0.3, V0=0.02, T=1
      alpha in {0.55, 0.62, 0.80, 0.95}
    """
    gamma = 0.3
    rho = -0.7
    nu = 1.0
    theta = 0.02 / 0.3
    V0 = 0.02
    T = 1.0
    eta = 2.0

    print("=== Slice-comparison rough cases ===")
    print(f"eta = {eta}")

    for alpha in [0.55, 0.62, 0.80, 0.95]:
        times, h = solve_fractional_riccati(alpha, gamma, rho, nu, eta + 0j, T=T, steps=4000)
        logPhi, _ = exact_discrete_transform_logphi(
            alpha=alpha,
            gamma=gamma,
            theta=theta,
            V0=V0,
            rho=rho,
            nu=nu,
            z=eta + 0j,
            T=T,
            tau=160,
        )
        print(
            f"alpha={alpha:.2f}, continuous h(T,eta)={h[-1]}, "
            f"log Phi^160(eta,T)={logPhi}"
        )


# ============================================================
# Classical Heston side (alpha = 1)
# ============================================================

def solve_heston_riccati_ode(
    gamma: float,
    rho: float,
    nu: float,
    z: complex,
    T: float = 1.0,
    steps: int = 20000,
    blowup_threshold: float = 1e8,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Classical Heston Riccati ODE for alpha=1:
        h' = 0.5(z^2-z) + gamma(rho*nu*z - 1) h + 0.5 (gamma*nu)^2 h^2, h(0)=0

    Simple RK4.
    """
    dt = T / steps
    times = np.linspace(0.0, T, steps + 1)
    h = np.zeros(steps + 1, dtype=complex)

    c0 = 0.5 * (z * z - z)
    c1 = gamma * (rho * nu * z - 1.0)
    c2 = 0.5 * (gamma * nu) ** 2

    def f(x: complex) -> complex:
        return c0 + c1 * x + c2 * x * x

    for n in range(steps):
        x = h[n]
        k1 = f(x)
        k2 = f(x + 0.5 * dt * k1)
        k3 = f(x + 0.5 * dt * k2)
        k4 = f(x + dt * k3)
        h[n + 1] = x + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)

        if not np.isfinite(h[n + 1].real) or not np.isfinite(h[n + 1].imag) or abs(h[n + 1]) > blowup_threshold:
            raise RuntimeError(f"Heston Riccati ODE appears to blow up before T={T}, at step {n+1}/{steps}.")

    return times, h


def check_classical_heston_case():
    """
    Classical benchmark with the first paper parameter set, alpha=1 only on continuous side.
    """
    gamma = 0.1
    rho = -0.681
    nu = 0.331
    theta = 0.3156
    V0 = 0.0392
    T = 1.0
    eta = 2.0

    print("=== Classical Heston benchmark ===")
    print(f"eta = {eta}")

    times, h = solve_heston_riccati_ode(gamma, rho, nu, eta + 0j, T=T)
    print(f"classical Heston h(T,eta) = {h[-1]}")


if __name__ == "__main__":
    check_first_rough_case()
    print()
    check_slice_comparison_case()
    print()
    check_classical_heston_case()

=== First rough case ===
eta = 2.0


continuous h(T,eta) = (0.9987830600699015+0j)
tau= 40, log Phi^tau(eta,T) = (0.056071711454221004+0j)
tau= 80, log Phi^tau(eta,T) = (0.0558250319766221+0j)
tau=160, log Phi^tau(eta,T) = (0.05569789300254024+0j)
tau=320, log Phi^tau(eta,T) = (0.055635943253161846+0j)

=== Slice-comparison rough cases ===
eta = 2.0


alpha=0.55, continuous h(T,eta)=(0.687781387384729+0j), log Phi^160(eta,T)=(0.023535195631845886+0j)


alpha=0.62, continuous h(T,eta)=(0.6970496905280966+0j), log Phi^160(eta,T)=(0.02329363706868555+0j)


alpha=0.80, continuous h(T,eta)=(0.7149051508841633+0j), log Phi^160(eta,T)=(0.02285083517941773+0j)


alpha=0.95, continuous h(T,eta)=(0.7207986036329125+0j), log Phi^160(eta,T)=(0.022581388552281378+0j)

=== Classical Heston benchmark ===
eta = 2.0
classical Heston h(T,eta) = (0.9310015437213601+0j)


In [2]:
import numpy as np
from math import gamma
from dataclasses import dataclass


@dataclass
class RHParams:
    alpha: float
    gamma_: float
    rho: float
    nu: float
    theta: float
    V0: float
    T: float = 1.0


def riccati_rhs(h: complex, z: complex, p: RHParams) -> complex:
    """
    RHS of the fractional Riccati equation:
        D^alpha h = a0 + a1*h + a2*h^2
    """
    a0 = 0.5 * (z * z - z)
    a1 = p.gamma_ * (p.rho * p.nu * z - 1.0)
    a2 = 0.5 * (p.gamma_ * p.nu) ** 2
    return a0 + a1 * h + a2 * h * h


def solve_fractional_riccati_pc(
    z: complex,
    p: RHParams,
    N: int = 2000,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Solve the Volterra form
        h(t) = 1/Gamma(alpha) * int_0^t (t-s)^(alpha-1) F(h(s), z) ds
    using a Diethelm-style PECE predictor-corrector scheme.

    Returns
    -------
    t : ndarray
        Time grid
    h : ndarray[complex]
        Approximate solution on the grid
    """
    alpha = p.alpha
    T = p.T
    dt = T / N
    t = np.linspace(0.0, T, N + 1, dtype=float)
    h = np.zeros(N + 1, dtype=np.complex128)

    # F(h_n, z)
    fvals = np.zeros(N + 1, dtype=np.complex128)
    fvals[0] = riccati_rhs(h[0], z, p)

    # Predictor weights:
    # b_{j,n+1} = (n+1-j)^alpha - (n-j)^alpha
    # h^P_{n+1} = dt^alpha / Gamma(alpha+1) * sum_{j=0}^n b_{j,n+1} f_j
    #
    # Corrector weights:
    # a_{0,n+1} = n^(alpha+1) - (n-alpha)(n+1)^alpha
    # a_{j,n+1} = (n-j+2)^(alpha+1) + (n-j)^(alpha+1) - 2(n-j+1)^(alpha+1), 1<=j<=n
    # a_{n+1,n+1} = 1
    # h_{n+1} = dt^alpha/Gamma(alpha+2) * [sum_{j=0}^n a_{j,n+1} f_j + f(h^P_{n+1})]

    g1 = gamma(alpha + 1.0)
    g2 = gamma(alpha + 2.0)

    for n in range(0, N):
        # predictor
        js = np.arange(0, n + 1)
        b = (n + 1 - js) ** alpha - (n - js) ** alpha
        h_pred = (dt ** alpha / g1) * np.sum(b * fvals[: n + 1])

        f_pred = riccati_rhs(h_pred, z, p)

        # corrector
        a = np.zeros(n + 1, dtype=float)
        if n == 0:
            a[0] = 1.0
        else:
            a[0] = (n ** (alpha + 1.0)) - (n - alpha) * ((n + 1) ** alpha)
            if n >= 1:
                jj = np.arange(1, n + 1)
                a[1:] = (
                    (n - jj + 2) ** (alpha + 1.0)
                    + (n - jj) ** (alpha + 1.0)
                    - 2.0 * (n - jj + 1) ** (alpha + 1.0)
                )

        h[n + 1] = (dt ** alpha / g2) * (np.sum(a * fvals[: n + 1]) + f_pred)
        fvals[n + 1] = riccati_rhs(h[n + 1], z, p)

    return t, h


def check_bounded_strip(
    p: RHParams,
    eta: float = 2.0,
    R: float = 20.0,
    n_xi: int = 81,
    N_time: int = 2000,
    blowup_threshold: float = 1e6,
    verbose: bool = True,
) -> dict:
    """
    Numerically check a sampled version of
        M_{eta,R,T} = sup_{|xi|<=R} sup_{0<=t<=T} |h(t, eta+i xi)| < infinity.

    On a discrete grid only.
    """
    xis = np.linspace(-R, R, n_xi)
    z_grid = eta + 1j * xis

    max_overall = 0.0
    worst_z = None
    max_per_z = []

    for z in z_grid:
        t, h = solve_fractional_riccati_pc(z, p, N=N_time)
        max_h = float(np.max(np.abs(h)))
        max_per_z.append(max_h)

        if np.any(~np.isfinite(h)) or max_h > blowup_threshold:
            return {
                "status": "warning",
                "message": f"Possible blow-up / instability detected at z={z}.",
                "eta": eta,
                "R": R,
                "T": p.T,
                "max_num": max_h,
                "worst_z": z,
                "xi_grid": xis,
                "max_per_z": np.array(max_per_z, dtype=float),
            }

        if max_h > max_overall:
            max_overall = max_h
            worst_z = z

    result = {
        "status": "ok",
        "eta": eta,
        "R": R,
        "T": p.T,
        "M_num": max_overall,
        "worst_z": worst_z,
        "xi_grid": xis,
        "max_per_z": np.array(max_per_z, dtype=float),
    }

    if verbose:
        print("=" * 72)
        print("Sampled bounded-strip Riccati check")
        print(f"alpha={p.alpha}, gamma={p.gamma_}, rho={p.rho}, nu={p.nu}, T={p.T}")
        print(f"eta={eta}, R={R}, n_xi={n_xi}, N_time={N_time}")
        print(f"M_num ≈ {max_overall:.10g}")
        print(f"worst sampled z = {worst_z}")
        print("=" * 72)

    return result


def refinement_test(
    p: RHParams,
    eta: float = 2.0,
    R: float = 20.0,
    n_xi: int = 81,
    N1: int = 1000,
    N2: int = 2000,
):
    """
    Compare coarse/fine time grids to get a crude stability check.
    """
    r1 = check_bounded_strip(p, eta=eta, R=R, n_xi=n_xi, N_time=N1, verbose=False)
    r2 = check_bounded_strip(p, eta=eta, R=R, n_xi=n_xi, N_time=N2, verbose=False)

    print("-" * 72)
    print("Refinement test")
    print(f"N_time={N1}: M_num ≈ {r1.get('M_num', np.nan):.10g}")
    print(f"N_time={N2}: M_num ≈ {r2.get('M_num', np.nan):.10g}")
    if r1["status"] == "ok" and r2["status"] == "ok":
        diff = abs(r2["M_num"] - r1["M_num"])
        print(f"|difference| ≈ {diff:.10g}")
    else:
        print("Warning: one of the runs signaled instability.")
    print("-" * 72)


def run_article_parameter_sets():
    # ------------------------------------------------------------------
    # Parameter set 1 from your paper
    # ------------------------------------------------------------------
    p1 = RHParams(
        alpha=0.62,
        gamma_=0.1,
        rho=-0.681,
        nu=0.331,
        theta=0.3156,
        V0=0.0392,
        T=1.0,
    )

    print("\n### Parameter set 1")
    check_bounded_strip(
        p1,
        eta=2.0,
        R=20.0,      # try also 10, 40, ...
        n_xi=81,
        N_time=2000,
    )
    refinement_test(
        p1,
        eta=2.0,
        R=20.0,
        n_xi=81,
        N1=1000,
        N2=2000,
    )

    # ------------------------------------------------------------------
    # Parameter set 2 from your slice-comparison experiment
    # gamma=0.3, rho=-0.7, nu=1, theta=0.02/0.3, V0=0.02, T=1
    # with alpha = 0.55, 0.62, 0.80, 0.95
    # ------------------------------------------------------------------
    base_kwargs = dict(
        gamma_=0.3,
        rho=-0.7,
        nu=1.0,
        theta=0.02 / 0.3,
        V0=0.02,
        T=1.0,
    )

    for alpha in [0.55, 0.62, 0.80, 0.95]:
        p2 = RHParams(alpha=alpha, **base_kwargs)

        print(f"\n### Parameter set 2, alpha={alpha}")
        check_bounded_strip(
            p2,
            eta=2.0,
            R=20.0,
            n_xi=81,
            N_time=2000,
        )
        refinement_test(
            p2,
            eta=2.0,
            R=20.0,
            n_xi=81,
            N1=1000,
            N2=2000,
        )


if __name__ == "__main__":
    run_article_parameter_sets()


### Parameter set 1


Sampled bounded-strip Riccati check
alpha=0.62, gamma=0.1, rho=-0.681, nu=0.331, T=1.0
eta=2.0, R=20.0, n_xi=81, N_time=2000
M_num ≈ 184.2933179
worst sampled z = (2-20j)


------------------------------------------------------------------------
Refinement test
N_time=1000: M_num ≈ 184.2932826
N_time=2000: M_num ≈ 184.2933179
|difference| ≈ 3.536772323e-05
------------------------------------------------------------------------

### Parameter set 2, alpha=0.55


Sampled bounded-strip Riccati check
alpha=0.55, gamma=0.3, rho=-0.7, nu=1.0, T=1.0
eta=2.0, R=20.0, n_xi=81, N_time=2000
M_num ≈ 53.31618639
worst sampled z = (2-20j)


------------------------------------------------------------------------
Refinement test
N_time=1000: M_num ≈ 53.31587875
N_time=2000: M_num ≈ 53.31618639
|difference| ≈ 0.0003076485774
------------------------------------------------------------------------

### Parameter set 2, alpha=0.62


...

------------------------------------------------------------------------
Refinement test
N_time=1000: M_num ≈ 54.10502595
N_time=2000: M_num ≈ 54.10522539
|difference| ≈ 0.0001994354408
------------------------------------------------------------------------

### Parameter set 2, alpha=0.8


Sampled bounded-strip Riccati check
alpha=0.8, gamma=0.3, rho=-0.7, nu=1.0, T=1.0
eta=2.0, R=20.0, n_xi=81, N_time=2000
M_num ≈ 56.49268155
worst sampled z = (2-20j)


------------------------------------------------------------------------
Refinement test
N_time=1000: M_num ≈ 56.492616
N_time=2000: M_num ≈ 56.49268155
|difference| ≈ 6.554522739e-05
------------------------------------------------------------------------

### Parameter set 2, alpha=0.95


Sampled bounded-strip Riccati check
alpha=0.95, gamma=0.3, rho=-0.7, nu=1.0, T=1.0
eta=2.0, R=20.0, n_xi=81, N_time=2000
M_num ≈ 59.06517762
worst sampled z = (2-20j)


------------------------------------------------------------------------
Refinement test
N_time=1000: M_num ≈ 59.06515234
N_time=2000: M_num ≈ 59.06517762
|difference| ≈ 2.528474543e-05
------------------------------------------------------------------------
